# 与 Pandas 的比较

Polars 和 Pandas 都是常用的数据处理库，但从设计上看，两者对“一张数据表是什么”的理解并不一样：Pandas 把行索引当作表结构的一部分，Polars 则完全不设行索引。这个差别会一路影响到运算的写法、性能表现和出错的方式。

这一章从行索引、列名、表达式、数据可变性、嵌套类型五个角度对比两者。读完之后应当能判断：一段 Pandas 代码搬到 Polars 上该怎么写，以及哪些习惯需要改掉。

In [1]:
import pandas as pd
import polars as pl
import numpy as np
from helper.jupyter import row

## 行索引

Pandas 用行索引(`Index` 对象)来按标签引用行，它是 DataFrame 结构的一部分，既用于按标签选行，也用于以索引为键把两个数据集对齐。默认是 0 开始的整数，但也可以指定任意标签，甚至用层次化的 `MultiIndex`。

按标签选行的 `df.loc[...]` 走哈希表，复杂度是 $O(1)$，因此需要反复按某个标签取行时，把它设成索引会快得多。

In [2]:
data = {"name": ["Alice", "Bob", "Charlie"], "age": [25, 30, 35], "index": ["a", "b", "c"]}
df = pd.DataFrame(data).set_index("index")
df.loc[["a"]]  # 走索引，所以是 O(1)

,name,age
index,,
a,Alice,25


Polars 没有行索引，行只能靠列的值来选：把条件传给 `filter()`，复杂度是 $O(N)$ 的线性扫描。

这样设计的好处是不依赖行号，内存布局和并行都更好优化；代价是“按标签取一行”这种操作在 Polars 里不再有捷径。如果行序或者行号本身有含义，请把它作为一列显式保存，而不是指望索引。

In [3]:
df = pl.DataFrame(data)
df.filter(pl.col("index") == "a")  # 按列的值查找，所以是 O(N)

name,age,index
str,i64,str
"""Alice""",25,"""a"""


两者的差别整理成下表。这里的“适用场景”不是绝对的好坏：数据量小的时候，用 Pandas 的索引反而更方便。

| 特征           | Pandas                     | Polars                     |
|----------------|----------------------------|----------------------------|
| **行索引**     | 有                         | 没有                       |
| **访问速度**   | $O(1)$(走索引时)         | $O(N)$(基于列的过滤)     |
| **设计取向**   | 重视行列操作的灵活性       | 重视列式与效率             |
| **适用场景**   | 小到中等规模的交互式处理   | 大规模数据的高速处理       |

Pandas 在计算前会先把行索引和列索引对齐，所以两个行顺序不同的表相减，也不会错位。

Polars 没有这套机制，同样的操作要显式做连接：下面的例子用 `df1` 减 `df2`，而 `df2` 只包含 `a`、`c`、`e` 三行。

In [4]:
df1 = pd.DataFrame(
    np.random.randint(0, 10, (5, 3)), 
    columns=['X', 'Y', 'Z'], 
    index=['a', 'b', 'c', 'd', 'e']
)
df2 = pd.DataFrame(
    np.random.randint(10, 20, (3, 2)), 
    columns=['X', 'Z'],
    index=['a', 'c', 'e']
)
row(df1, df2, df1.sub(df2, fill_value=0))

X 
 Y 
 Z 
 
 
 
 
 a 
 8 
 1 
 9 
 
 
 b 
 1 
 0 
 0 
 
 
 c 
 6 
 9 
 7 
 
 
 d 
 4 
 2 
 3 
 
 
 e 
 7 
 4 
 9 
 
 
 
 

 
 
 
 
 X 
 Z 
 
 
 
 
 a 
 13 
 18 
 
 
 c 
 14 
 11 
 
 
 e 
 14 
 12 
 
 
 
 

 
 
 
 
 X 
 Y 
 Z 
 
 
 
 
 a 
 -5.0 
 1.0 
 -9.0 
 
 
 b 
 1.0 
 0.0 
 0.0 
 
 
 c 
 -8.0 
 9.0 
 -4.0 
 
 
 d 
 4.0 
 2.0 
 3.0 
 
 
 e 
 -7.0 
 4.0 
 -3.0

要用 Polars 做同样的事，就得先用 `join()` 把行对齐，再按列套用计算式。这里写一个通用的 `align_op()`。

因为用了 `LazyFrame`，连接、填空、列运算会合成一个查询，交给 Rust 侧优化后再整体执行，而不是一步一步跑三遍。

In [5]:
def align_op(df1, df2, op, on="index", fill_value=0):
    # 只对两边都有的列做运算（结合键除外）
    common_columns = [c for c in df1.columns if c in df2.columns and c != on]
    return (
        df1.lazy()
        .join(df2.lazy(), on=on, how="left", maintain_order="left")
        .fill_null(fill_value)  # df2 里没有的行按 0 处理
        .with_columns(
            [
                op(pl.col(col), pl.col(f"{col}_right")).alias(col)
                for col in common_columns
            ]
        )
        .select(df1.columns)
        .collect()
    )


# 把 Pandas 的索引落到列上，再交给 Polars
df3 = pl.from_pandas(df1.reset_index())
df4 = pl.from_pandas(df2.reset_index())
row(df3, df4, align_op(df3, df4, pl.Expr.sub))

shape: (5, 4) index X Y Z str i32 i32 i32 "a" 8 1 9 "b" 1 0 0 "c" 6 9 7 "d" 4 2 3 "e" 7 4 9 
 shape: (3, 3) index X Z str i32 i32 "a" 13 18 "c" 14 11 "e" 14 12 
 shape: (5, 4) index X Y Z str i32 i32 i32 "a" -5 1 -9 "b" 1 0 0 "c" -8 9 -4 "d" 4 2 3 "e" -7 4 -3

## 列名的差别

Pandas 允许列名重复，也能一次选出同名的多列：下面的例子里 `age` 出现两次，`df2[["age"]]` 会把两列都取出来。这种表在做完透视或拼接之后很常见，但在 Polars 里会直接报错。

In [6]:
df = pd.DataFrame(data).set_index('index')
df2 = df[['name', 'age', 'age']]
row(df2, df2[['age']])

name 
 age 
 age 
 
 
 index 
 
 
 
 
 
 
 
 a 
 Alice 
 25 
 25 
 
 
 b 
 Bob 
 30 
 30 
 
 
 c 
 Charlie 
 35 
 35 
 
 
 
 

 
 
 
 
 age 
 age 
 
 
 index 
 
 
 
 
 
 
 a 
 25 
 25 
 
 
 b 
 30 
 30 
 
 
 c 
 35 
 35

Polars 不允许列名重复，想用两次同一列就必须用 `alias()` 起别名。下面的代码把第二个 `age` 改成 `age_2`，再用正则 `^age.*$` 把以 `age` 开头的列一起选出来。

正则选择器是 Polars 补偿“不能重名”的常用手段：名不能一样，但可以按模式批量选。

In [7]:
df = pl.DataFrame(data)
df2 = df.select('name', 'age', pl.col('age').alias('age_2'))
row(df2, df2.select(pl.col('^age.*$')))

shape: (3, 3) name age age_2 str i64 i64 "Alice" 25 25 "Bob" 30 30 "Charlie" 35 35 
 shape: (3, 2) age age_2 i64 i64 25 25 30 30 35 35

Pandas 的列名可以是整数或元组，这样能把多级表头放进一个 DataFrame。灵活是灵活，但列名类型不统一时，代码里到处是 `df[10]` 这样的魔法值，读起来很难受。

In [8]:
data = {10: ['Alice', 'Bob', 'Charlie'], 20: [25, 30, 35]}
df = pd.DataFrame(data)
df[10]

0      Alice
1        Bob
2    Charlie
Name: 10, dtype: str

Polars 的列名必须是字符串。类型固定下来，表达式和方法的参数就好写也好读；想用整数键，就先转成字符串再传进去。

In [9]:
df = pl.DataFrame({str(key):value for key, value in data.items()})
df.select(pl.col('10'))

10
str
"""Alice"""
"""Bob"""
"""Charlie"""


## 表达式

表达式(expression)表示“对列的计算”，它本身不是结果。Polars 先搭好表达式，再交给 `select()` 或 `with_columns()`，到这时才真正求值。

正因为计算被推迟了，优化器才有机会把整条查询合并、剪掉用不到的部分；Pandas 没有这个概念，写下的操作立刻执行，也不会跨语句优化；

In [10]:
df = pd.DataFrame(dict(
    x=[1, 2, 3],
    y=[4, 5, 6]
))

df['dist'] = (df['x']**2 + df['y']**2)**0.5
df

,x,y,dist
0,1,4,4.123106
1,2,5,5.385165
2,3,6,6.708204


Pandas 想把计算合在一起写，也不是没办法：可以把表达式写成字符串交给 `eval()`；但它支持的语法有限，遇到自定义函数就只能退回逐条执行；

In [11]:
df['dist2'] = df.eval('(x**2 + y**2)**0.5')
df

,x,y,dist,dist2
0,1,4,4.123106,4.123106
1,2,5,5.385165,5.385165
2,3,6,6.708204,6.708204


Polars 的写法是把表达式传给 `select()` 或 `with_columns()`，调用那一刻才开始求值。

In [12]:
df = pl.DataFrame(dict(
    x=[1, 2, 3],
    y=[4, 5, 6]
))

df.with_columns(
    dist = (pl.col('x')**2 + pl.col('y')**2).sqrt()
)

x,y,dist
i64,i64,f64
1,4,4.123106
2,5,5.385165
3,6,6.708204


`LazyFrame` 把求值再往后推一步：调用 `select()`、`with_columns()` 时只是往计划里加节点，直到 `collect()` 才执行。

Polars 2.0 的 `collect()` 默认走流式引擎，所以内存装不下的数据也能分块算完。

In [13]:
(df
.lazy()
.select(
    x2 = pl.col('x')**2,
    y2 = pl.col('y')**2
)
.select(
    xy2 = pl.col('x2') + pl.col('y2')
)
.select(
    dist = pl.col('xy2').sqrt()
)
.collect()
)

dist
f64
4.123106
5.385165
6.708204


用表达式有三个好处。
1. **延迟求值** — 表达式定义时不执行，等到需要结果才算，所以用不到的列不会被白算一遍。
2. **链式** — 多个表达式可以串起来写，复杂的处理也能从上往下读，不必给每一步都起中间变量名。
3. **可优化** — Polars 先看整条查询再定执行计划，表达式分开写也能在一次遍历里完成。

## 数据不可变

Pandas 里常用 `inplace=True` 或按标签赋值来就地更新：下面的例子直接把符合条件的那几行的工资改掉，`df` 本身被改了。

In [14]:
data = {"name": ["Alice", "Bob", "Charlie"], "age": [25, 30, 35], "salary": [50000, 60000, 70000]}
df = pd.DataFrame(data)

# 选出年龄 30 以上的人，工资加 5000
df.loc[df["age"] >= 30, "salary"] += 5000
df

,name,age,salary
0,Alice,25,50000
1,Bob,30,65000
2,Charlie,35,75000


Polars 的 DataFrame 是不可变的：任何“修改”都返回一个新表，原表不动。

In [15]:
df = pl.DataFrame(data)

# 定义表达式：年龄 30 以上时工资加 5000
expr = (
    pl.when(pl.col("age") >= 30)
    .then(pl.col("salary") + 5000)
    .otherwise(pl.col("salary"))
)

# 套用表达式，得到新表
df2 = df.with_columns(expr.alias("updated_salary"))
row(df, df2)

shape: (3, 3) name age salary str i64 i64 "Alice" 25 50000 "Bob" 30 60000 "Charlie" 35 70000 
 shape: (3, 4) name age salary updated_salary str i64 i64 i64 "Alice" 25 50000 50000 "Bob" 30 60000 65000 "Charlie" 35 70000 75000

这不只是风格问题，它决定了代码怎么调试、怎么复用。
1. **安全** — 原始数据不变，函数调用不会意外改掉调用方的表。
2. **好调试** — 每一步的表都留着，能追到值究竟在哪一步变了。
3. **和链式天然契合** — 每个操作都返回新表，所以可以直接连成一条链，不必先复制再修改。

## 嵌套类型

Polars 原生支持列表(`List`)和结构体(`Struct`)，而且可以任意嵌套，所以“一行里带一个列表”这种数据不用拆表就能存。

下面的代码里 `attrs` 是结构体列，其中的 `score` 字段又是列表，❶ 的表达式直接对这个列表求平均。

In [16]:
df = pl.DataFrame({
    "name": ['A', 'B', 'C'],
    "attrs": [
        {"age": 30, "score":[89, 90, 100]},
        {"age": 25, "score":[60]},
        {"age": 35, "score":[70, 80]},
    ]
})

df.select(
    "name",
    mean_score=pl.col("attrs").struct.field("score").list.mean() #❶
)

name,mean_score
str,f64
"""A""",93.0
"""B""",60.0
"""C""",75.0


Pandas 没有对应的类型系统，嵌套数据只能放进 `object` 列，每个元素都是 Python 对象。

算的时候要逐个元素进 Python 层，行数一多就慢；下面这段代码里，`attrs` 的每个元素都是字典。

In [17]:
import numpy as np

df = pd.DataFrame({
    "name": ['A', 'B', 'C'],
    "attrs": [
        {"age": 30, "score":[89, 90, 100]},
        {"age": 25, "score":[60]},
        {"age": 35, "score":[70, 80]},
    ]
})
df['mean_score'] = df['attrs'].str['score'].apply(np.mean)
df

,name,attrs,mean_score
0,A,"{'age': 30, 'score': [89, 90, 100]}",93.0
1,B,"{'age': 25, 'score': [60]}",60.0
2,C,"{'age': 35, 'score': [70, 80]}",75.0
